In [1]:
%env NODE_OPTIONS=--max-old-space-size=32768

env: NODE_OPTIONS=--max-old-space-size=32768


In [5]:
%cd ..
%cd ..
%cd ..

/sise/home/urizlo/VuLLM_One_Stage


In [11]:
from code_files.preprocess_data import Prepare_dataset_with_only_replace_only_encoder
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [12]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-7b-instruct-v1.5", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-7b-instruct-v1.5", trust_remote_code=True, torch_dtype=torch.bfloat16).cuda()


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [54]:
path_trainset = "Datasets/vulgen_train_with_diff_lines_spaces.csv"
path_testset = "Datasets/vulgen_test_with_diff_lines_spaces.csv"
full_vulgen = True
train, test = Prepare_dataset_with_only_replace_only_encoder.create_datasets(path_trainset, path_testset, full_vulgen=full_vulgen)
train['prompt'] = train.apply(lambda row: f"""Return the modified vulnerable version of this function without explnations \nfunction:\n{row['inputs']}""", axis=1)
test['prompt'] = train.apply(lambda row: f"""Return the modified vulnerable version of this function without explnations \nfunction:\n{row['inputs']}""", axis=1)

8135
978


In [55]:
print(test['prompt'].iloc[0])

Return the modified vulnerable version of this function without explnations 
function:
int net_get(int s, void *arg, int *len) {
    struct net_hdr nh;
    int plen;
    if (net_read_exact(s, &nh, sizeof(nh)) == -1) {
        return -1; }
    plen = ntohl(nh.nh_len);
    if (!(plen <= *len))
        printf("PLEN %d type %d len %d\n", plen, nh.nh_type, *len);
    assert(plen <= *len && plen > 0);
    *len = plen;
    if ((*len) && (net_read_exact(s, arg, *len) == -1)) {
        return -1; }  
    return nh.nh_type; }


In [57]:
messages=[
    { 'role': 'user', 'content': test['prompt'].iloc[0]}
]
# inputs = tokenizer(train['prompt'].iloc[0], return_tensors="pt", padding="max_length", truncation=True)
# inputs = inputs.to("cuda")  # Move to CUDA if you're using GPU
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = model.config.eos_token_id
outputs = model.generate(inputs, max_new_tokens=512, num_return_sequences=1, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True))

```c
int net_get(int s, void *arg, int *len) {
    struct net_hdr nh;
    int plen;
    if (net_read_exact(s, &nh, sizeof(nh)) == -1) {
        return -1; }
    plen = ntohl(nh.nh_len);
    if (!(plen <= *len))
        printf("PLEN %d type %d len %d\n", plen, nh.nh_type, *len);
    assert(plen <= *len && plen > 0);
    *len = plen;
    if ((*len) && (net_read_exact(s, arg, *len) == -1)) {
        return -1; }  
    return nh.nh_type; }
```

